In [1]:
from pyspark.sql import SparkSession

spark = SparkSession.builder.appName("lab3").getOrCreate()
spark.sparkContext.setLogLevel("WARN")
spark

In [2]:
temperature_sensor = (
    (12.5, "2019-01-02 12:00:00"),
    (17.6, "2019-01-02 12:00:20"),
    (14.6, "2019-01-02 12:00:30"),
    (22.9, "2019-01-02 12:01:15"),
    (17.4, "2019-01-02 12:01:30"),
    (25.8, "2019-01-02 12:03:25"),
    (27.1, "2019-01-02 12:02:40"),
)

In [3]:
from pyspark.sql.functions import to_timestamp
from pyspark.sql.types import StructType, StructField, StringType, DoubleType

schema = StructType([
    StructField("temperature", DoubleType(), True),
    StructField("time", StringType(), True),
])

In [4]:
df = (
    spark.createDataFrame(temperature_sensor, schema=schema)
    .withColumn("time", to_timestamp("time"))
)

df.printSchema()
df.show(3)

root
 |-- temperature: double (nullable = true)
 |-- time: timestamp (nullable = true)

+-----------+-------------------+
|temperature|               time|
+-----------+-------------------+
|       12.5|2019-01-02 12:00:00|
|       17.6|2019-01-02 12:00:20|
|       14.6|2019-01-02 12:00:30|
+-----------+-------------------+
only showing top 3 rows



In [5]:
df.createOrReplaceTempView("df")

spark.sql("""
SELECT time, temperature
FROM df
WHERE temperature > 21
""").show()

+-------------------+-----------+
|               time|temperature|
+-------------------+-----------+
|2019-01-02 12:01:15|       22.9|
|2019-01-02 12:03:25|       25.8|
|2019-01-02 12:02:40|       27.1|
+-------------------+-----------+



In [ ]:
# 2. Batch tumbling window

In [6]:
import pyspark.sql.functions as F

df2 = df.groupBy(F.window("time", "30 seconds")).count()
df2.show(truncate=False)
df2.printSchema()

+------------------------------------------+-----+
|window                                    |count|
+------------------------------------------+-----+
|{2019-01-02 12:00:00, 2019-01-02 12:00:30}|2    |
|{2019-01-02 12:00:30, 2019-01-02 12:01:00}|1    |
|{2019-01-02 12:01:00, 2019-01-02 12:01:30}|1    |
|{2019-01-02 12:01:30, 2019-01-02 12:02:00}|1    |
|{2019-01-02 12:03:00, 2019-01-02 12:03:30}|1    |
|{2019-01-02 12:02:30, 2019-01-02 12:03:00}|1    |
+------------------------------------------+-----+

root
 |-- window: struct (nullable = false)
 |    |-- start: timestamp (nullable = true)
 |    |-- end: timestamp (nullable = true)
 |-- count: long (nullable = false)



In [ ]:
# 3. Rate stream test

In [7]:
from pyspark.sql import SparkSession

spark = SparkSession.builder.appName("StreamingDemo").getOrCreate()
spark.sparkContext.setLogLevel("WARN")

def process_batch(df, batch_id, tstop=5):
    print(f"Batch ID: {batch_id}")
    df.show(truncate=False)
    if batch_id >= tstop:
        raise Exception("stop")

df_stream = (
    spark.readStream
    .format("rate")
    .option("rowsPerSecond", 1)
    .load()
)

query = (
    df_stream.writeStream
    .outputMode("append")
    .foreachBatch(process_batch)
    .option("truncate", False)
    .start()
)

try:
    query.awaitTermination()
except:
    query.stop()

Batch ID: 0
+---------+-----+
|timestamp|value|
+---------+-----+
+---------+-----+

Batch ID: 1
+-----------------------+-----+
|timestamp              |value|
+-----------------------+-----+
|2026-05-04 13:48:46.365|0    |
+-----------------------+-----+

Batch ID: 2
+-----------------------+-----+
|timestamp              |value|
+-----------------------+-----+
|2026-05-04 13:48:47.365|1    |
+-----------------------+-----+

Batch ID: 3
+-----------------------+-----+
|timestamp              |value|
+-----------------------+-----+
|2026-05-04 13:48:48.365|2    |
+-----------------------+-----+

Batch ID: 4
+-----------------------+-----+
|timestamp              |value|
+-----------------------+-----+
|2026-05-04 13:48:49.365|3    |
+-----------------------+-----+

Batch ID: 5
+-----------------------+-----+
|timestamp              |value|
+-----------------------+-----+
|2026-05-04 13:48:50.365|4    |
+-----------------------+-----+

Batch ID: 0
+----+-----------+
|time|temperature|


In [8]:
from pyspark.sql.functions import col, expr

stream = (
    df_stream
    .withColumn("time", col("timestamp"))
    .withColumn("temperature", expr("20 + rand() * 10"))
    .select("time", "temperature")
)

query = (
    stream.writeStream
    .outputMode("append")
    .foreachBatch(process_batch)
    .option("truncate", False)
    .start()
)

try:
    query.awaitTermination()
except:
    query.stop()

In [9]:
stream_filtered = stream.filter(col("temperature") > 28)

query = (
    stream_filtered.writeStream
    .outputMode("append")
    .foreachBatch(process_batch)
    .option("truncate", False)
    .start()
)

try:
    query.awaitTermination()
except:
    query.stop()

In [10]:
from pyspark.sql.functions import when

stream_anomaly = stream.withColumn(
    "anomaly",
    when(col("temperature") > 29.5, "YES").otherwise("NO")
)

query = (
    stream_anomaly.writeStream
    .outputMode("append")
    .foreachBatch(process_batch)
    .option("truncate", False)
    .start()
)

try:
    query.awaitTermination()
except:
    query.stop()

In [11]:
from pyspark.sql.functions import window

tumbling_window = (
    stream
    .groupBy(window(col("time"), "10 seconds"))
    .avg("temperature")
)

query = (
    tumbling_window.writeStream
    .outputMode("complete")
    .foreachBatch(process_batch)
    .option("truncate", False)
    .start()
)

try:
    query.awaitTermination()
except:
    query.stop()

In [12]:
sliding = (
    stream
    .groupBy(window(col("time"), "10 seconds", "5 seconds"))
    .avg("temperature")
)

query = (
    sliding.writeStream
    .outputMode("complete")
    .foreachBatch(process_batch)
    .option("truncate", False)
    .start()
)

try:
    query.awaitTermination()
except:
    query.stop()

In [13]:
%%file generator.py
import json, os, random, time
from datetime import datetime

output_dir = "data/stream"
os.makedirs(output_dir, exist_ok=True)

stores = ['Warsaw', 'Krakow', 'Gdansk', 'Wroclaw']
categories = ['electronics', 'clothing', 'food', 'books']

def generate_transaction():
    return {
        'tx_id': f'TX{random.randint(1000, 9999)}',
        'user_id': f'u{random.randint(1, 20):02d}',
        'amount': round(random.uniform(5.0, 5000.0), 2),
        'store': random.choice(stores),
        'category': random.choice(categories),
        'timestamp': datetime.now().isoformat(),
    }

while True:
    batch = [generate_transaction() for _ in range(2)]
    filename = f"{output_dir}/events_{int(time.time())}.json"
    with open(filename, "w") as f:
        for e in batch:
            f.write(json.dumps(e) + "\n")
    print(f"Wrote: {filename}")
    time.sleep(5)

Writing generator.py


In [14]:
from pyspark.sql.types import StructType, StructField, StringType, DoubleType

tx_schema = StructType([
    StructField("tx_id", StringType()),
    StructField("user_id", StringType()),
    StructField("amount", DoubleType()),
    StructField("store", StringType()),
    StructField("category", StringType()),
    StructField("timestamp", StringType()),
])

batch_counter = {"count": 0}

def process_batch(df, batch_id, tstop=5):
    batch_counter["count"] += 1
    print(f"Batch ID: {batch_id}")
    df.show(truncate=False)
    if batch_counter["count"] >= tstop:
        raise Exception("stop")

file_stream = (
    spark.readStream
    .schema(tx_schema)
    .json("data/stream")
)

query = (
    file_stream.writeStream
    .foreachBatch(process_batch)
    .start()
)

try:
    query.awaitTermination()
except:
    query.stop()

In [15]:
spark.stop()